# 🔍 Error Analysis

Analyze failure cases and OCR errors.


In [ ]:
import sys
sys.path.append('../src')

from evaluation.metrics import plate_accuracy, edit_distance, character_accuracy
from storage.database import PlateDatabase
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.style.use('dark_background')

db = PlateDatabase()
logs = db.get_logs(limit=1000)
df = pd.DataFrame(logs)
print(f'Total records: {len(df)}')
if not df.empty:
    print(df.head())


In [ ]:
# Confidence distribution
if not df.empty and 'confidence' in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(df['confidence'], bins=20, color='cyan', edgecolor='black')
    axes[0].set_title('Confidence Distribution')
    axes[0].set_xlabel('Confidence')
    axes[0].set_ylabel('Count')

    valid_counts = df['is_valid'].value_counts()
    axes[1].pie(valid_counts.values, labels=['Valid', 'Invalid'][:len(valid_counts)],
                colors=['#00FF88', '#FF4757'], autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Valid vs Invalid Plates')

    if 'source_type' in df.columns:
        df['source_type'].value_counts().plot(kind='bar', ax=axes[2], color='steelblue')
        axes[2].set_title('Source Type Distribution')
        axes[2].set_xlabel('')

    plt.tight_layout()
    plt.savefig('../reports/figures/error_analysis.png', dpi=150)
    plt.show()


In [ ]:
# Most common provinces
if not df.empty and 'province_name' in df.columns:
    province_counts = df['province_name'].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(12, 5))
    province_counts.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Top 15 Provinces in Detection Logs')
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.savefig('../reports/figures/province_distribution.png', dpi=150)
    plt.show()


In [ ]:
# OCR character error analysis using past evaluations
eval_runs = db.get_evaluations(limit=10)
if eval_runs:
    df_eval = pd.DataFrame(eval_runs)
    print('\n=== Evaluation History ===')
    print(df_eval[['timestamp', 'dataset', 'exact_match_rate', 'char_accuracy', 'total_samples']].to_string())
else:
    print('No evaluation runs found. Run evaluations in the Streamlit UI.')
